# Fusion — Phase 3/4

**Goal:** Combine the audio track output (MIDI notes + timing) with the vision track output (finger positions per frame) to produce a final string/fret assignment for every note.

**This notebook assumes:**
- You have run `audio_track.ipynb` and have `audio_notes` (list of NoteEvent dicts)
- You have run `vision_track.ipynb` and have `vision_frames` (list of per-frame dicts)
- You have implemented at minimum `aitabs.mapping.fingering.map_sequence` (DP fallback)

The final output is a `.gp5` file you can open in Guitar Pro or TuxGuitar.

Stages:
1. Load audio and vision outputs
2. Run fusion (resolve string/fret for each note)
3. Measure audio-only vs fusion accuracy against ground truth
4. Export to .gp5

## 0. Imports and config

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

# These will exist once you have run the other notebooks and saved their outputs.
# For now, define placeholder variables:
audio_notes   = []   # list of NoteEvent from aitabs.audio.pitch.detect_notes
vision_frames = []   # list of frame dicts from the vision track notebook
VIDEO_FPS     = 30.0

print('Placeholder variables set — replace with real outputs from audio/vision notebooks')

## 1. Audio-only baseline (DP mapping)

Before the vision layer is ready, get the audio-only pipeline producing real GP5 output. This:
- Establishes your baseline accuracy number
- Lets you verify the GP5 export works end-to-end
- Gives you something to demo while the vision layer is in progress

The DP fingering optimizer (`map_sequence`) assigns string/fret using minimum-movement heuristics without any visual information. This is roughly what audio-only competitors like Klangio do.

In [ ]:
# TODO: uncomment after implementing aitabs/mapping/fingering.py

# from aitabs.mapping.fingering import map_sequence

# audio_only_notes = map_sequence(audio_notes)

# import librosa
# STRING_NAMES = ['E2', 'A2', 'D3', 'G3', 'B3', 'e4']
# print(f'Mapped {len(audio_only_notes)} notes (audio-only DP)')
# for note in audio_only_notes[:15]:
#     name = librosa.midi_to_note(note['pitch_midi'])
#     print(f"{name:>5}  {note['start']:>6.2f}s  {STRING_NAMES[note['string']]}  fret={note['fret']}")

print('Uncomment after implementing map_sequence')

## 2. Fusion (audio + vision)

After the vision track is working, replace the audio-only mapping with the fusion layer.

**Expected improvement:** The vision layer resolves string ambiguity. For example, the MIDI note E3 (52) can be played on:
- String 0 (low E), fret 12
- String 1 (A), fret 7
- String 2 (D), fret 2

Audio alone can't determine which — the DP heuristic guesses based on context. Vision can see which string the finger is actually on and resolve it directly.

In [ ]:
# TODO: uncomment after implementing aitabs/fusion/fusion.py

# from aitabs.fusion.fusion import fuse

# fused_notes = fuse(audio_notes, vision_frames, video_fps=VIDEO_FPS)

# sources = {note['source'] for note in fused_notes}
# for source in sorted(sources):
#     count = sum(1 for n in fused_notes if n['source'] == source)
#     pct   = 100 * count / len(fused_notes)
#     print(f'  {source:<20} {count:>4} notes ({pct:.0f}%)')

print('Uncomment after implementing fuse')

## 3. Accuracy measurement

**This section is the most important one for the YC application.**

Ground truth format: for each recording session, you write the correct tab by hand (or by ear) into a CSV:
```
start_sec, pitch_midi, string, fret
0.12, 64, 5, 0
0.45, 62, 4, 3
...
```

**Metrics to track:**
- **Note accuracy:** % of detected notes where (string, fret) exactly matches ground truth, within a 50 ms timing window.
- **Pitch accuracy:** % of detected notes where MIDI pitch matches ground truth (audio-only measure, ignores string/fret).
- **False positive rate:** Detected notes that have no matching ground truth note.
- **False negative rate:** Ground truth notes that have no matching detection.

**Target:** 80% note accuracy on clean solo recordings by end of summer.

In [ ]:
# TODO: implement after you have ground truth CSV files

def compute_accuracy(predicted: list, ground_truth: list, timing_window: float = 0.05):
    """Compute note accuracy: % of predicted notes that match ground truth.
    
    A note is a match if:
      - pitch_midi matches exactly
      - string matches exactly  
      - fret matches exactly
      - |predicted_start - gt_start| <= timing_window
    
    TODO: implement this for your accuracy tracking spreadsheet.
    """
    raise NotImplementedError

print('compute_accuracy stub defined — implement before your first accuracy measurement')

## 4. Export to Guitar Pro (.gp5)

Once the notes are assigned string/fret, export to GP5. Open the output in Guitar Pro or TuxGuitar (free) to visually verify that the tab looks correct.

**Visual checks:**
- Do the fret numbers look like something a human would play?
- Are notes on reasonable strings (no jumps from low E to high e between consecutive notes)?
- Does the rhythm look approximately correct (even if quantization is rough)?

In [ ]:
# TODO: uncomment after implementing aitabs/output/gp5.py

# from aitabs.output.gp5 import export_gp5

# OUTPUT_GP5 = '../output/test_output.gp5'
# import os; os.makedirs('../output', exist_ok=True)

# export_gp5(fused_notes, OUTPUT_GP5, tempo_bpm=120)
# print(f'Exported to {OUTPUT_GP5}')

print('Uncomment after implementing export_gp5')

## 5. Weekly accuracy log

Keep this table updated. The YC application will want to see this improving over time.

| Date | Recording | Notes | Pitch Acc. | Note Acc. (audio-only) | Note Acc. (fusion) |
|------|-----------|-------|------------|----------------------|-------------------|
| 2026-06-01 | test_01 | — | — | — | — |

Fill in `—` as you complete each phase.